In [7]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# 1. 列名清理函数（解决f-string反斜杠语法错误）
def clean_column_name(col_name):
    """清理列名中的换行符和空格，避免格式问题"""
    return str(col_name).replace("\n", "").strip()

# 2. 读取Excel文件（指定你的文件路径+兼容xlsx格式）
def load_excel_data(file_path):
    """读取表单数据和受教育经历表，解决xlrd不支持xlsx的问题"""
    try:
        # 明确指定engine='openpyxl'，确保能读取xlsx格式
        df_main = pd.read_excel(file_path, sheet_name='表单数据', engine='openpyxl')
        df_education = pd.read_excel(file_path, sheet_name='受教育经历（从高中开始）', engine='openpyxl')
        print(f"✅ 成功读取文件：{file_path}")
        print(f" - 表单数据表：{df_main.shape[0]}行 × {df_main.shape[1]}列")
        print(f" - 受教育经历表：{df_education.shape[0]}行 × {df_education.shape[1]}列")
        return df_main, df_education
    except Exception as e:
        print(f"❌ 读取文件失败：{str(e)}")
        print("请检查：1.文件路径是否正确 2.Excel文件是否处于关闭状态 3.工作表名称是否匹配")
        raise  # 抛出错误以便定位问题

# 3. 数据预处理（清理无效数据+统一格式）
def preprocess_data(df_main, df_education):
    """清理说明行、空值，统一学工号格式，确保匹配准确性"""
    # 清理表单数据：删除第一行说明行+去除学工号为空的记录
    df_main_clean = df_main.iloc[1:].copy()  # 去掉Excel首行的"上传文件时请删除本行"说明
    df_main_clean = df_main_clean[df_main_clean['学工号'].notna()].copy()  # 过滤学工号为空的无效行
    df_main_clean['学工号'] = df_main_clean['学工号'].astype(str).str.strip()  # 统一学工号为字符串（避免数字格式差异）
    
    # 清理受教育经历表：同样删除说明行+过滤空值
    df_education_clean = df_education.iloc[1:].copy()
    df_education_clean = df_education_clean[df_education_clean['学工号'].notna()].copy()
    df_education_clean['学工号'] = df_education_clean['学工号'].astype(str).str.strip()
    
    # 自动识别受教育经历的核心字段（学校、入学时间、毕业时间、学历层次等）
    education_fields = []
    for col in df_education_clean.columns:
        col_clean = clean_column_name(col)
        # 基于关键词匹配需要整合的字段，确保不遗漏关键信息
        if any(keyword in col_clean for keyword in ['学校', '入学', '毕业', '学历', '专业', '全日制']):
            education_fields.append(col)
    
    print(f"\n🔍 识别到受教育经历核心字段：{[clean_column_name(col) for col in education_fields]}")
    return df_main_clean, df_education_clean, education_fields

# 4. 整合受教育经历数据（处理一对多关系）
def integrate_education_data(df_main, df_education, education_columns):
    """将受教育经历表的多条记录按学工号聚合，用|分隔，合并到表单数据"""
    def aggregate_education(group):
        """聚合同一学生的多条受教育经历，避免数据重复"""
        aggregated_data = {}
        for col in education_columns:
            # 去重+去空值，只保留有效记录
            non_null_vals = group[col].dropna().astype(str).unique()
            valid_vals = [val for val in non_null_vals if val.strip() != '']
            # 多条经历用|分隔（例："XX高中|XX大学本科"），无数据则填"无"
            aggregated_data[col] = '|'.join(valid_vals) if len(valid_vals) > 0 else '无'
        return pd.Series(aggregated_data)
    
    # 按学工号分组聚合，确保1个学生只对应1行数据
    df_education_agg = df_education.groupby('学工号', group_keys=False).apply(aggregate_education).reset_index()
    
    # 重命名受教育经历字段（添加"受教育经历_"前缀，便于区分来源）
    education_rename_map = {}
    for col in education_columns:
        clean_col = clean_column_name(col)
        education_rename_map[col] = f'受教育经历_{clean_col}'  # 避免字段名重复
    
    df_education_agg = df_education_agg.rename(columns=education_rename_map)
    
    # 左连接合并到表单数据：确保不丢失任何学生记录
    df_integrated = pd.merge(
        df_main,
        df_education_agg,
        on='学工号',
        how='left'  # 左连接：保留表单数据所有学生，无经历的学生字段填"无"
    )
    
    # 填充空值：无受教育经历的学生字段统一填"无"，避免NaN
    for new_col in education_rename_map.values():
        if new_col in df_integrated.columns:
            df_integrated[new_col] = df_integrated[new_col].fillna('无')
    
    print(f"\n📊 数据整合完成：")
    print(f" - 最终表单数据：{df_integrated.shape[0]}行 × {df_integrated.shape[1]}列")
    print(f" - 新增受教育经历字段：{list(education_rename_map.values())}")
    return df_integrated, education_rename_map

# 5. 保存文件+统一表格格式（行高/列宽一致）
def save_integrated_data(df_integrated, input_path, education_rename_map):
    """保存整合后的数据，设置统一行高、列宽和视觉样式"""
    # 生成输出路径：和原文件同目录，文件名添加"_整合受教育经历"后缀
    output_path = input_path.replace("批量操作模板.xlsx", "批量操作模板_整合受教育经历.xlsx")
    
    # 保存数据到Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_integrated.to_excel(writer, sheet_name='表单数据_整合后', index=False)
    
    # 加载文件设置格式：确保每一行每一列高度宽度一致
    wb = load_workbook(output_path)
    ws = wb['表单数据_整合后']
    
    # 1. 统一行高：22磅（适配1-2行文字，避免内容截断）
    UNIFORM_ROW_HEIGHT = 22
    for row_num in range(1, ws.max_row + 1):
        ws.row_dimensions[row_num].height = UNIFORM_ROW_HEIGHT
    
    # 2. 统一列宽：16字符（适配中文字段名和数据，避免显示不全）
    UNIFORM_COL_WIDTH = 16
    for col_num in range(1, ws.max_column + 1):
        col_letter = ws.cell(row=1, column=col_num).column_letter
        ws.column_dimensions[col_letter].width = UNIFORM_COL_WIDTH
    
    # 3. 样式优化（提升可读性，非必需但体验更好）
    # 表头样式：蓝色背景+白色粗体+居中
    header_font = Font(bold=True, color="FFFFFF")
    header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    
    # 数据样式：居中对齐+自动换行
    data_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    
    # 边框样式：灰色细边框（便于区分单元格）
    thin_border = Border(
        left=Side(style='thin', color="CCCCCC"),
        right=Side(style='thin', color="CCCCCC"),
        top=Side(style='thin', color="CCCCCC"),
        bottom=Side(style='thin', color="CCCCCC")
    )
    
    # 应用表头样式
    for cell in ws[1]:  # 第一行是表头
        cell.font = header_font
        cell.alignment = header_alignment
        cell.fill = header_fill
        cell.border = thin_border
    
    # 应用数据样式
    for row_num in range(2, ws.max_row + 1):  # 从第二行开始是数据
        for col_num in range(1, ws.max_column + 1):
            cell = ws.cell(row=row_num, column=col_num)
            cell.alignment = data_alignment
            cell.border = thin_border
    
    # 保存格式设置
    wb.save(output_path)
    print(f"\n🎉 全部流程完成！")
    print(f"📁 输出文件路径：{output_path}")
    print(f"✅ 已实现：1.受教育经历数据整合 2.统一行高{UNIFORM_ROW_HEIGHT}磅 3.统一列宽{UNIFORM_COL_WIDTH}字符")

# 6. 主函数（串联所有步骤，直接执行）
def main():
    # 你的文件路径（已按你提供的路径设置，无需修改）
    input_file_path = r"D:\桌面\批量操作模板.xlsx"  # r前缀处理路径中的反斜杠，避免转义错误
    
    try:
        print("="*60)
        print("开始执行受教育经历数据整合流程")
        print("="*60)
        
        # 步骤1：读取Excel文件
        df_main, df_education = load_excel_data(input_file_path)
        
        # 步骤2：数据预处理
        df_main_clean, df_education_clean, education_fields = preprocess_data(df_main, df_education)
        
        # 步骤3：整合受教育经历数据到表单数据
        df_integrated, education_rename_map = integrate_education_data(df_main_clean, df_education_clean, education_fields)
        
        # 步骤4：保存并设置统一格式
        save_integrated_data(df_integrated, input_file_path, education_rename_map)
        
    except Exception as e:
        print(f"\n❌ 流程执行失败：{str(e)}")
        print("常见解决办法：")
        print(" 1. 关闭Excel文件（避免文件被占用）")
        print(" 2. 检查路径是否存在：D:\\桌面\\批量操作模板.xlsx")
        print(" 3. 确认Excel中有'表单数据'和'受教育经历（从高中开始）'两个工作表")

# 执行主函数（运行代码的入口）
if __name__ == "__main__":
    main()

开始执行受教育经历数据整合流程
❌ 读取文件失败：Unknown engine: openpyxl
请检查：1.文件路径是否正确 2.Excel文件是否处于关闭状态 3.工作表名称是否匹配

❌ 流程执行失败：Unknown engine: openpyxl
常见解决办法：
 1. 关闭Excel文件（避免文件被占用）
 2. 检查路径是否存在：D:\桌面\批量操作模板.xlsx
 3. 确认Excel中有'表单数据'和'受教育经历（从高中开始）'两个工作表


In [8]:
def load_excel_data(file_path):
    """读取Excel文件（自动适配引擎，不依赖openpyxl）"""
    try:
        # 🔥 关键修改：去掉 engine='openpyxl'，让pandas自动选择
        df_main = pd.read_excel(file_path, sheet_name='表单数据')
        df_education = pd.read_excel(file_path, sheet_name='受教育经历（从高中开始）')
        print(f"✅ 成功读取文件：{file_path}")
        print(f" - 表单数据表：{df_main.shape[0]}行 × {df_main.shape[1]}列")
        print(f" - 受教育经历表：{df_education.shape[0]}行 × {df_education.shape[1]}列")
        return df_main, df_education
    except Exception as e:
        print(f"❌ 读取文件失败：{str(e)}")
        raise

In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# 1. 列名清理函数
def clean_column_name(col_name):
    return str(col_name).replace("\n", "").strip()

# 2. 读取Excel文件（修复点：去掉engine参数）
def load_excel_data(file_path):
    try:
        df_main = pd.read_excel(file_path, sheet_name='表单数据')
        df_education = pd.read_excel(file_path, sheet_name='受教育经历（从高中开始）')
        print(f"✅ 成功读取文件！")
        return df_main, df_education
    except Exception as e:
        print(f"❌ 读取失败：{str(e)}")
        raise

# 3. 数据预处理
def preprocess_data(df_main, df_education):
    df_main_clean = df_main.iloc[1:].copy()
    df_main_clean = df_main_clean[df_main_clean['学工号'].notna()].copy()
    df_main_clean['学工号'] = df_main_clean['学工号'].astype(str).str.strip()
    
    df_education_clean = df_education.iloc[1:].copy()
    df_education_clean = df_education_clean[df_education_clean['学工号'].notna()].copy()
    df_education_clean['学工号'] = df_education_clean['学工号'].astype(str).str.strip()
    
    education_fields = []
    for col in df_education_clean.columns:
        col_clean = clean_column_name(col)
        if any(keyword in col_clean for keyword in ['学校', '入学', '毕业', '学历', '专业']):
            education_fields.append(col)
    
    return df_main_clean, df_education_clean, education_fields

# 4. 整合数据
def integrate_education_data(df_main, df_education, education_columns):
    def aggregate_education(group):
        agg = {}
        for col in education_columns:
            vals = group[col].dropna().astype(str).unique()
            vals = [v for v in vals if v.strip()]
            agg[col] = '|'.join(vals) if vals else '无'
        return pd.Series(agg)
    
    edu_agg = df_education.groupby('学工号', group_keys=False).apply(aggregate_education).reset_index()
    
    rename_map = {col: f'受教育经历_{clean_column_name(col)}' for col in education_columns}
    edu_agg = edu_agg.rename(columns=rename_map)
    
    merged = pd.merge(df_main, edu_agg, on='学工号', how='left')
    
    for new_col in rename_map.values():
        if new_col in merged.columns:
            merged[new_col] = merged[new_col].fillna('无')
    
    return merged, rename_map

# 5. 保存与格式设置
def save_and_format(df_integrated, input_path):
    output_path = input_path.replace(".xlsx", "_整合受教育经历.xlsx")
    
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df_integrated.to_excel(writer, sheet_name='整合后', index=False)
    
    wb = load_workbook(output_path)
    ws = wb.active
    
    # 统一行高列宽
    for row in range(1, ws.max_row + 1):
        ws.row_dimensions[row].height = 22
    for col in range(1, ws.max_column + 1):
        ws.column_dimensions[ws.cell(row=1, column=col).column_letter].width = 16
    
    # 样式
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.fill = header_fill
        cell.border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
    
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))
    
    wb.save(output_path)
    print(f"🎉 成功生成：{output_path}")

# 6. 主入口
def main():
    file_path = r"D:\桌面\批量操作模板.xlsx"
    try:
        df_main, df_edu = load_excel_data(file_path)
        df_main_clean, df_edu_clean, edu_cols = preprocess_data(df_main, df_edu)
        df_merged, rename_map = integrate_education_data(df_main_clean, df_edu_clean, edu_cols)
        save_and_format(df_merged, file_path)
    except Exception as e:
        print(f"❌ 最终失败：{str(e)}")

if __name__ == "__main__":
    main()

✅ 成功读取文件！
🎉 成功生成：D:\桌面\批量操作模板_整合受教育经历.xlsx
